## Random Forest

In [65]:
%pip install fastparquet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error, classification_report
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
from pathlib import Path

root = next(p for p in Path.cwd().parents if (p / "config.yml").exists())

panel = pd.read_parquet(
    root / "data/processed_data/analysis_panel.parquet", engine='fastparquet'
)
print(panel.shape)
print(panel.dtypes)
panel.head(20)

In [15]:

# ─────────────────────────────────────────────
# 1. CARGAR DATOS
# ─────────────────────────────────────────────
root = next(p for p in Path.cwd().parents if (p / "config.yml").exists())

df = pd.read_parquet(
    root / "data/processed_data/analysis_panel.parquet", engine='fastparquet'
)
print(panel.shape)
print(panel.dtypes)
panel.head(20)

# Filtrar solo 2022
df = df[df["YEAR"] == 2022].copy()
print(f"Shape filtrado (2022): {df.shape}")
print("Sectores IS8:", df["IS8_SECTOR"].unique())

(17150, 57)
YEAR                               int16
GEOGRAPHY_CODE                  category
GEOGRAPHY_NAME                  category
IS8_SECTOR                      category
EMPLOYEES                        float32
BUSINESSES                       float32
gva_per_hour                     float64
weekly_pay                       float64
employment_rate                  float64
unemployment_rate                float64
gdhi_per_head                    float64
new_enterprises                  float64
deaths_of_enterprises            float64
active_enterprises               float64
high_growth_enterprises          float64
public_transport_to_employer     float64
drive_to_employer                float64
cycle_to_employer                float64
broadband_availability           float64
4g_area_coverage                 float64
ks2_attainment                   float64
gcse_by_age_19                   float64
ofsted                           float64
persistent_absences              float64
pers

In [28]:

# ─────────────────────────────────────────────
# 2. DEFINIR VARIABLES
# ─────────────────────────────────────────────

# Variables dependientes
dep_vars_regression     = ["lq_emp", "lq_bus", "growth_emp", "growth_bus"]
dep_vars_classification = ["lq_emp", "lq_bus", "growth_emp", "growth_bus"]  # binarizadas con umbral >= mediana del sector

# Variables explicativas (nombres reales del parquet)
features = [
    # Human capital
    "level_3+_qualifications",      # nvq_level3
    "gcse_by_age_19",               # gcse_age19
    "apprenticeship_starts",
    "apprenticeship_achievements",
    "fe_and_skills_participation",  # fe_participation

    # Entrepreneurial discovery
    "new_enterprises",              # enterprise_births
    "deaths_of_enterprises",        # enterprise_deaths
    "active_enterprises",           # enterprise_active
    "high_growth_enterprises",      # enterprise_high_growth

    # Connectivity
    "public_transport_to_employer", # transport_to_employer
    "drive_to_employer",
    "cycle_to_employer",
    "broadband_availability",       # broadband
    "4g_area_coverage",             # coverage_4g

    # Labour market
    "unemployment_rate",

    # Place conditions
    "net_additions",                # housing_net_additions

    # Derived
    "related_variety",
    # business_density no está como columna — se puede calcular si hay población
]

# Verificar presencia en el dataframe
missing  = [f for f in features if f not in df.columns]
features = [f for f in features if f in df.columns]

if missing:
    print(f"\n⚠️  Columnas no encontradas (se ignorarán): {missing}\n")
print(f"\nFeatures usados ({len(features)}): {features}\n")




Features usados (17): ['level_3+_qualifications', 'gcse_by_age_19', 'apprenticeship_starts', 'apprenticeship_achievements', 'fe_and_skills_participation', 'new_enterprises', 'deaths_of_enterprises', 'active_enterprises', 'high_growth_enterprises', 'public_transport_to_employer', 'drive_to_employer', 'cycle_to_employer', 'broadband_availability', '4g_area_coverage', 'unemployment_rate', 'net_additions', 'related_variety']



In [30]:

# ─────────────────────────────────────────────
# 3. CONFIGURACIÓN
# ─────────────────────────────────────────────
RF_PARAMS = dict(n_estimators=300, max_depth=None, min_samples_leaf=3,
                 n_jobs=-1, random_state=42)
CV = KFold(n_splits=5, shuffle=True, random_state=42)

sectors   = sorted(df["IS8_SECTOR"].dropna().unique())
results   = []
fi_store  = []
pred_store = []

# ─────────────────────────────────────────────
# 4. LOOP PRINCIPAL
# ─────────────────────────────────────────────
for sector in sectors:
    sub = df[df["IS8_SECTOR"] == sector].copy()
    X   = sub[features].copy()
    X   = X.fillna(X.median(numeric_only=True))

    print(f"\n{'='*60}")
    print(f"  SECTOR: {sector}  |  n = {len(sub)}")
    print(f"{'='*60}")

    # ── 4A. REGRESIÓN ────────────────────────────────────────────
    for dep in dep_vars_regression:
        if dep not in sub.columns:
            continue
        y    = sub[dep].copy()
        mask = y.notna()
        X_r, y_r = X[mask], y[mask]

        if len(y_r) < 20:
            print(f"  [SKIP regresión {dep}] n={len(y_r)} < 20")
            continue

        model = RandomForestRegressor(**RF_PARAMS)
        model.fit(X_r, y_r)

        r2_cv   = cross_val_score(model, X_r, y_r, cv=CV, scoring="r2")
        rmse_cv = np.sqrt(-cross_val_score(model, X_r, y_r, cv=CV,
                          scoring="neg_mean_squared_error"))

        print(f"\n  [Regresión] {dep}")
        print(f"    R²   CV: {r2_cv.mean():.3f} ± {r2_cv.std():.3f}")
        print(f"    RMSE CV: {rmse_cv.mean():.4f} ± {rmse_cv.std():.4f}")

        fi_store.append(pd.DataFrame({
            "sector": sector, "dep_var": dep, "model_type": "regression",
            "feature": features, "importance": model.feature_importances_
        }))

        results.append({
            "sector": sector, "dep_var": dep, "model_type": "regression",
            "n": int(mask.sum()),
            "r2_cv_mean":   round(r2_cv.mean(), 4),
            "r2_cv_std":    round(r2_cv.std(), 4),
            "rmse_cv_mean": round(rmse_cv.mean(), 4),
            "rmse_cv_std":  round(rmse_cv.std(), 4),
        })

        pred_df = sub[mask][["IS8_SECTOR", "GEOGRAPHY_CODE", "GEOGRAPHY_NAME", "YEAR"]].copy()
        pred_df["dep_var"]    = dep
        pred_df["model_type"] = "regression"
        pred_df["y_true"]     = y_r.values
        pred_df["y_pred"]     = model.predict(X_r)
        pred_store.append(pred_df)

    # ── 4B. CLASIFICACIÓN ────────────────────────────────────────
    for dep in dep_vars_classification:
        if dep not in sub.columns:
            continue
        y_cont = sub[dep].copy()
        mask   = y_cont.notna()
        X_c    = X[mask]
        # LQ: umbral natural = 1 (especializado vs no)
        # growth: umbral = mediana del sector (crecimiento alto vs bajo)
        threshold = 1 if dep in ("lq_emp", "lq_bus") else y_cont[mask].median()
        y_c    = (y_cont[mask] >= threshold).astype(int)
        dep_clf = f"{dep}_binary"

        if len(y_c) < 20 or y_c.nunique() < 2:
            print(f"  [SKIP clasificación {dep_clf}] n<20 o clase única")
            continue

        model_c = RandomForestClassifier(**RF_PARAMS)
        model_c.fit(X_c, y_c)

        acc_cv = cross_val_score(model_c, X_c, y_c, cv=CV, scoring="accuracy")
        f1_cv  = cross_val_score(model_c, X_c, y_c, cv=CV, scoring="f1")

        umbral_str = "LQ ≥ 1" if dep in ("lq_emp", "lq_bus") else f"growth ≥ mediana ({threshold:.3f})"
        print(f"\n  [Clasificación] {dep_clf}  (umbral: {umbral_str})")
        print(f"    Accuracy CV: {acc_cv.mean():.3f} ± {acc_cv.std():.3f}")
        print(f"    F1       CV: {f1_cv.mean():.3f} ± {f1_cv.std():.3f}")
        print(classification_report(y_c, model_c.predict(X_c),
              target_names=["no especializado", "especializado"]))

        fi_store.append(pd.DataFrame({
            "sector": sector, "dep_var": dep_clf, "model_type": "classification",
            "feature": features, "importance": model_c.feature_importances_
        }))

        results.append({
            "sector": sector, "dep_var": dep_clf, "model_type": "classification",
            "n": int(mask.sum()),
            "acc_cv_mean": round(acc_cv.mean(), 4),
            "acc_cv_std":  round(acc_cv.std(), 4),
            "f1_cv_mean":  round(f1_cv.mean(), 4),
            "f1_cv_std":   round(f1_cv.std(), 4),
        })

        pred_df_c = sub[mask][["IS8_SECTOR", "GEOGRAPHY_CODE", "GEOGRAPHY_NAME", "YEAR"]].copy()
        pred_df_c["dep_var"]    = dep_clf
        pred_df_c["model_type"] = "classification"
        pred_df_c["y_true"]     = y_c.values
        pred_df_c["y_pred"]     = model_c.predict(X_c)
        pred_store.append(pred_df_c)



  SECTOR: Advanced Manufacturing  |  n = 350

  [Regresión] lq_emp
    R²   CV: 0.121 ± 0.136
    RMSE CV: 1.2028 ± 0.2673

  [Regresión] lq_bus
    R²   CV: 0.391 ± 0.048
    RMSE CV: 0.3494 ± 0.0328

  [Regresión] growth_emp
    R²   CV: -0.198 ± 0.261
    RMSE CV: 0.4333 ± 0.1720

  [Regresión] growth_bus
    R²   CV: 0.060 ± 0.092
    RMSE CV: 0.2812 ± 0.0743

  [Clasificación] lq_emp_binary  (umbral: LQ ≥ 1)
    Accuracy CV: 0.691 ± 0.077
    F1       CV: 0.661 ± 0.075
                  precision    recall  f1-score   support

no especializado       0.96      0.99      0.98       194
   especializado       0.99      0.96      0.97       156

        accuracy                           0.98       350
       macro avg       0.98      0.97      0.98       350
    weighted avg       0.98      0.98      0.98       350


  [Clasificación] lq_bus_binary  (umbral: LQ ≥ 1)
    Accuracy CV: 0.743 ± 0.024
    F1       CV: 0.712 ± 0.047
                  precision    recall  f1-score   suppor

In [41]:

# ─────────────────────────────────────────────
# 5. EXPORTAR
# ─────────────────────────────────────────────
results_df = pd.DataFrame(results)
fi_df      = pd.concat(fi_store, ignore_index=True)
preds_df   = pd.concat(pred_store, ignore_index=True)

results_df.to_csv("rf_metrics.csv", index=False)
fi_df.to_csv("rf_feature_importance.csv", index=False)
preds_df.to_csv("rf_predictions.csv", index=False)

print("\n✅ Archivos exportados:")
print("   rf_metrics.csv")
print("   rf_feature_importance.csv")
print("   rf_predictions.csv")

# ─────────────────────────────────────────────
# 6. RESUMEN
# ─────────────────────────────────────────────
print("\n── MÉTRICAS RESUMEN ──────────────────────────────────────")
print(results_df.to_string(index=False))




✅ Archivos exportados:
   rf_metrics.csv
   rf_feature_importance.csv
   rf_predictions.csv

── MÉTRICAS RESUMEN ──────────────────────────────────────
                            sector           dep_var     model_type   n  r2_cv_mean  r2_cv_std  rmse_cv_mean  rmse_cv_std  acc_cv_mean  acc_cv_std  f1_cv_mean  f1_cv_std
            Advanced Manufacturing            lq_emp     regression 350      0.1207     0.1358        1.2028       0.2673          NaN         NaN         NaN        NaN
            Advanced Manufacturing            lq_bus     regression 350      0.3907     0.0485        0.3494       0.0328          NaN         NaN         NaN        NaN
            Advanced Manufacturing        growth_emp     regression 350     -0.1981     0.2605        0.4333       0.1720          NaN         NaN         NaN        NaN
            Advanced Manufacturing        growth_bus     regression 349      0.0598     0.0916        0.2812       0.0743          NaN         NaN         NaN        N

In [44]:
# summary for lq_emp regression

print("\n── TOP 5 FEATURES por sector (regresión lq_emp) ──────────")
top_fi_lqemp = (
    fi_df[(fi_df["dep_var"] == "lq_emp") & (fi_df["model_type"] == "regression")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_lqemp.to_string(index=False))


── TOP 5 FEATURES por sector (regresión lq_emp) ──────────
                            sector                      feature  importance
            Advanced Manufacturing                net_additions    0.193521
            Advanced Manufacturing  apprenticeship_achievements    0.133203
            Advanced Manufacturing            unemployment_rate    0.087157
            Advanced Manufacturing        apprenticeship_starts    0.084238
            Advanced Manufacturing              related_variety    0.073376
               Creative Industries public_transport_to_employer    0.344499
               Creative Industries        apprenticeship_starts    0.193604
               Creative Industries            cycle_to_employer    0.100045
               Creative Industries      level_3+_qualifications    0.059428
               Creative Industries               gcse_by_age_19    0.049106
                           Defence       broadband_availability    0.123245
                           D

In [45]:
# summary for lq_bus regression

print("\n── TOP 5 FEATURES por sector (regression lq_bus) ──────────")
top_fi_lqbus = (
    fi_df[(fi_df["dep_var"] == "lq_bus") & (fi_df["model_type"] == "regression")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_lqbus.to_string(index=False))


── TOP 5 FEATURES por sector (regression lq_bus) ──────────
                            sector                      feature  importance
            Advanced Manufacturing        apprenticeship_starts    0.199863
            Advanced Manufacturing              related_variety    0.165710
            Advanced Manufacturing  apprenticeship_achievements    0.113092
            Advanced Manufacturing      level_3+_qualifications    0.104981
            Advanced Manufacturing  fe_and_skills_participation    0.058791
               Creative Industries        apprenticeship_starts    0.424985
               Creative Industries  apprenticeship_achievements    0.188625
               Creative Industries public_transport_to_employer    0.080276
               Creative Industries      level_3+_qualifications    0.069293
               Creative Industries               gcse_by_age_19    0.045187
                           Defence                net_additions    0.297375
                           

In [46]:
# Summary for classification - lq_emp

print("\n── TOP 5 FEATURES por sector (classification lq_emp) ──────────")
top_fi_class_lqemp = (
    fi_df[(fi_df["dep_var"] == "lq_emp_binary") & (fi_df["model_type"] == "classification")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_class_lqemp.to_string(index=False))


── TOP 5 FEATURES por sector (classification lq_emp) ──────────
                            sector                     feature  importance
            Advanced Manufacturing       apprenticeship_starts    0.114731
            Advanced Manufacturing apprenticeship_achievements    0.097392
            Advanced Manufacturing fe_and_skills_participation    0.091034
            Advanced Manufacturing     level_3+_qualifications    0.079923
            Advanced Manufacturing             related_variety    0.063072
               Creative Industries       apprenticeship_starts    0.145529
               Creative Industries     level_3+_qualifications    0.145362
               Creative Industries apprenticeship_achievements    0.106406
               Creative Industries              gcse_by_age_19    0.100106
               Creative Industries     high_growth_enterprises    0.077299
                           Defence       apprenticeship_starts    0.110201
                           Defence 

In [47]:
# Summary for classification - businesses

print("\n── TOP 5 FEATURES por sector (classification lq_bus) ──────────")
top_fi_class_lqbus = (
    fi_df[(fi_df["dep_var"] == "lq_bus_binary") & (fi_df["model_type"] == "classification")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_class_lqbus.to_string(index=False))


── TOP 5 FEATURES por sector (classification lq_bus) ──────────
                            sector                     feature  importance
            Advanced Manufacturing       apprenticeship_starts    0.119348
            Advanced Manufacturing             related_variety    0.117204
            Advanced Manufacturing     level_3+_qualifications    0.105923
            Advanced Manufacturing apprenticeship_achievements    0.089452
            Advanced Manufacturing            4g_area_coverage    0.059506
               Creative Industries       apprenticeship_starts    0.256283
               Creative Industries apprenticeship_achievements    0.156700
               Creative Industries     level_3+_qualifications    0.127495
               Creative Industries              gcse_by_age_19    0.107167
               Creative Industries           cycle_to_employer    0.051688
                           Defence       apprenticeship_starts    0.226711
                           Defence 

# For growth

In [48]:
# Summary regresion growth_emp

print("\n── TOP 5 FEATURES por sector (regression growth_emp) ──────────")
top_fi_gr_emp = (
    fi_df[(fi_df["dep_var"] == "growth_emp") & (fi_df["model_type"] == "regression")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_gr_emp.to_string(index=False))


── TOP 5 FEATURES por sector (regression growth_emp) ──────────
                            sector                      feature  importance
            Advanced Manufacturing               gcse_by_age_19    0.181180
            Advanced Manufacturing       broadband_availability    0.120571
            Advanced Manufacturing              new_enterprises    0.064201
            Advanced Manufacturing        apprenticeship_starts    0.056974
            Advanced Manufacturing  apprenticeship_achievements    0.054808
               Creative Industries              related_variety    0.106237
               Creative Industries      level_3+_qualifications    0.080899
               Creative Industries  fe_and_skills_participation    0.071164
               Creative Industries                net_additions    0.068066
               Creative Industries      high_growth_enterprises    0.067954
                           Defence        apprenticeship_starts    0.326293
                       

In [49]:
# Summary regresion growth_bus

print("\n── TOP 5 FEATURES por sector (regression growth_bus) ──────────")
top_fi_gr_bus = (
    fi_df[(fi_df["dep_var"] == "growth_bus") & (fi_df["model_type"] == "regression")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_gr_bus.to_string(index=False))


── TOP 5 FEATURES por sector (regression growth_bus) ──────────
                            sector                      feature  importance
            Advanced Manufacturing              related_variety    0.156221
            Advanced Manufacturing               gcse_by_age_19    0.101785
            Advanced Manufacturing        deaths_of_enterprises    0.100238
            Advanced Manufacturing            cycle_to_employer    0.073032
            Advanced Manufacturing public_transport_to_employer    0.072463
               Creative Industries              related_variety    0.321926
               Creative Industries  fe_and_skills_participation    0.074639
               Creative Industries      high_growth_enterprises    0.068439
               Creative Industries      level_3+_qualifications    0.064338
               Creative Industries       broadband_availability    0.051459
          Digital and Technologies      high_growth_enterprises    0.226892
          Digital and T

In [51]:
# Summary classification growth_emp

print("\n── TOP 5 FEATURES por sector (regression growth_emp) ──────────")
top_fi_class_gremp = (
    fi_df[(fi_df["dep_var"] == "growth_emp_binary") & (fi_df["model_type"] == "classification")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_class_gremp.to_string(index=False))


── TOP 5 FEATURES por sector (regression growth_emp) ──────────
                            sector                      feature  importance
            Advanced Manufacturing            unemployment_rate    0.082772
            Advanced Manufacturing           active_enterprises    0.076717
            Advanced Manufacturing       broadband_availability    0.074958
            Advanced Manufacturing              new_enterprises    0.068172
            Advanced Manufacturing        deaths_of_enterprises    0.065588
               Creative Industries              related_variety    0.077279
               Creative Industries       broadband_availability    0.064726
               Creative Industries public_transport_to_employer    0.064311
               Creative Industries      level_3+_qualifications    0.063775
               Creative Industries            cycle_to_employer    0.063417
                           Defence               gcse_by_age_19    0.101432
                       

In [52]:
# Summary classification growth_emp

print("\n── TOP 5 FEATURES por sector (regression growth_bus) ──────────")
top_fi_class_grbus = (
    fi_df[(fi_df["dep_var"] == "growth_bus_binary") & (fi_df["model_type"] == "classification")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_class_grbus.to_string(index=False))


── TOP 5 FEATURES por sector (regression growth_bus) ──────────
                            sector                      feature  importance
            Advanced Manufacturing              related_variety    0.109924
            Advanced Manufacturing  fe_and_skills_participation    0.083347
            Advanced Manufacturing           active_enterprises    0.070282
            Advanced Manufacturing               gcse_by_age_19    0.067959
            Advanced Manufacturing      level_3+_qualifications    0.066454
               Creative Industries              related_variety    0.141240
               Creative Industries      level_3+_qualifications    0.073196
               Creative Industries  fe_and_skills_participation    0.069723
               Creative Industries               gcse_by_age_19    0.066996
               Creative Industries       broadband_availability    0.060885
          Digital and Technologies              related_variety    0.156084
          Digital and T